# Long-Running and Asynchronous Agents

## Scenario: durable EU checkout investigation

This job lasts beyond a request: it waits for deployment evidence, pauses for an incident commander, recovers after worker loss, and completes only after revalidation. **Safety boundary:** a resume never grants new authority; production action remains outside this lab.

![Long-running agent durable lifecycle](../../../assets/long-running-agent-lifecycle.svg)

Persist job/tenant/owner, state schema, deadline, budgets, allowed actions, idempotency key, event correlation ID, approval fingerprint, cancellation, and audit state. A wait means a scheduler or trusted event will wake the job—not a live model polling loop.

## 1. State machine and checkpoint rules

Checkpoint after every durable transition. Pause before a side effect. On resume, re-check tenant, reviewer identity, policy, freshness, exact-action fingerprint, deadline, and idempotency. Reads may be retryable; writes require server-side idempotency and reconciliation. Treat duplicate, late, and out-of-order events as normal delivery conditions.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'advanced' / '10-long-running-asynchronous-agents'
sys.path.insert(0, str(TOPIC))
from lab import Job, start, resume, recover

job = Job('job-42', 'acme')
start(job)
print(job.state, job.audit)
assert resume(job, 'evidence-ready') == 'approval-requested'
job = recover(job)
assert resume(job, 'approved') == 'complete'
print(job.state, job.audit)

waiting-for-evidence ['checkpoint:waiting-for-evidence']
complete ['checkpoint:waiting-for-evidence', 'checkpoint:waiting-for-approval', 'recovered-from-checkpoint', 'checkpoint:complete']


## 2. Background, scheduled, and event-driven runs

Use a background queue for durable work, a scheduler for periodic work, and authenticated event delivery for external wakes. Every route needs a correlation ID, dedupe/idempotency policy, expiry, rate/spend/concurrency budget, cancellation path, retention policy, and telemetry. Use human approval when the next step is consequential; approval applies to one immutable proposal only.

In [2]:
# Deliberate failure: a job must expire instead of accepting a late event forever.
late = Job('late-1', 'acme', deadline_step=0)
start(late)
assert resume(late, 'evidence-ready') == 'expired'
assert resume(late, 'approved') == 'ignored'
print(late.state, late.audit)

expired ['checkpoint:waiting-for-evidence', 'checkpoint:expired']


## Production checklist and exercises

Validate producer/authentication/schema/tenant/replay for events; use leases and heartbeats; persist checkpoints; apply explicit time, action, retry, concurrency, and spend budgets; expose pause/cancel/retry/escalate; test worker loss, duplicate/late events, approval expiry, cancellation races, migrations, and outages; and evaluate recovery correctness, duplicate avoidance, p95 end-to-end time, and cost per completed safe job.

**Exercises:** add duplicate approval protection, design a v1→v2 state migration, implement a cancellation race test, and compare a trusted event wait to an unsafe polling loop.

References: [LangGraph durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution), [Temporal workflows](https://docs.temporal.io/workflows), [OpenAI agent guide](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/).